# Lecture 10 Example — End-to-End Review

**Course:** AAU E26 — Introduction to Scripting, Data Mining and Machine Learning  
**Lecture:** Lecture 10  
**Goal:** Review the complete computational and data-analysis process in one compact case.

[Open this notebook in Google Colab](https://colab.research.google.com/github/asmrabbi/E26_TAN7_Scripting_CPH/blob/main/notebooks/examples/L10_example_end_to_end_review.ipynb) · [View the course repository](https://github.com/asmrabbi/E26_TAN7_Scripting_CPH)

Run the cells from top to bottom. Every executable line includes a short comment explaining what it does.


## Goal

Review the complete course workflow in a compact, executable case and prepare to explain each decision orally.


## Setup

This notebook uses the synthetic monthly service report and repeats only the operations needed for the final summary.


## Steps

### 1. Load, preserve, and prepare


In [1]:
from pathlib import Path  # Imports Path so the notebook can find a local course file when available.
import pandas as pd  # Imports pandas for reading and working with table-shaped data.
remote_data_url = "https://raw.githubusercontent.com/asmrabbi/E26_TAN7_Scripting_CPH/main/data/monthly_service_report.csv"  # Stores the public GitHub address used by Google Colab.
local_data_candidates = [Path("data/monthly_service_report.csv"), Path("../../data/monthly_service_report.csv")]  # Lists possible local paths used during validation.
data_source = next((path for path in local_data_candidates if path.exists()), remote_data_url)  # Chooses a local file when present and otherwise uses GitHub.
raw_report = pd.read_csv(data_source)  # Reads the CSV file into a pandas DataFrame.
print(raw_report.head())  # Prints a small preview so we can confirm that loading worked.


   record_id report_month          city service_type  cases_received  \
0       1001   2026-01-01    Copenhagen      Housing             120   
1       1002   2026-01-01   copenhagen     Transport              85   
2       1003   2026-01-01       AALBORG      Housing              -3   
3       1004   2026-02-01    Koebenhavn   Employment              74   
4       1004   2026-02-01    Koebenhavn   Employment              74   

   cases_resolved resolution_days  satisfaction_score  \
0             112             5.1                 4.2   
1              90             3.2                 4.6   
2               0             8.4                 3.1   
3              68             4.0                 4.0   
4              68             4.0                 4.0   

                                  feedback  
0           Helpful staff and clear answer  
1  Quick answer but the form was confusing  
2                  Long wait for an answer  
3                   The guidance was clear  

In [2]:
import matplotlib.pyplot as plt  # Imports plotting tools for the final figure.
prepared_report = raw_report.drop_duplicates().copy()  # Removes confirmed exact duplicates while preserving the raw DataFrame.
prepared_report["city"] = prepared_report["city"].astype("string").str.strip().str.lower().replace({"koebenhavn": "copenhagen"}).str.title()  # Standardises the known city variants.
prepared_report["cases_received"] = pd.to_numeric(prepared_report["cases_received"], errors="coerce")  # Converts received-case values and exposes invalid text as missing.
prepared_report["cases_resolved"] = pd.to_numeric(prepared_report["cases_resolved"], errors="coerce")  # Converts resolved-case values and exposes invalid text as missing.
prepared_report["satisfaction_score"] = pd.to_numeric(prepared_report["satisfaction_score"], errors="coerce")  # Converts satisfaction scores and exposes invalid text as missing.
prepared_report["valid_counts"] = (prepared_report["cases_received"] >= 0) & (prepared_report["cases_resolved"] >= 0) & (prepared_report["cases_resolved"] <= prepared_report["cases_received"])  # Applies the stated simplified validity rule.


### 2. Summarise only valid rows


In [3]:
valid_report = prepared_report[prepared_report["valid_counts"]].copy()  # Creates an analytical subset while retaining invalid rows in the prepared audit table.
city_results = valid_report.groupby("city").agg(records=("record_id", "count"), received=("cases_received", "sum"), resolved=("cases_resolved", "sum"), mean_satisfaction=("satisfaction_score", "mean")).reset_index()  # Produces one row of transparent summaries per city.
city_results["resolution_rate"] = city_results["resolved"] / city_results["received"]  # Calculates a city-level resolution proportion.
print(city_results.round(3))  # Displays the bounded final results table.


         city  records  received  resolved  mean_satisfaction  resolution_rate
0     Aalborg        6       506       463              3.933            0.915
1  Copenhagen        7       680       643              4.517            0.946


### 3. Visualise the descriptive comparison


In [4]:
plt.figure(figsize=(7, 4))  # Creates a readable figure for the city comparison.
plt.bar(city_results["city"], city_results["resolution_rate"], color="#005AA0")  # Draws one bar per city using the calculated rate.
plt.title("Resolution rate in valid teaching records")  # States the measure and the important subset restriction.
plt.xlabel("City")  # Labels the categorical horizontal axis.
plt.ylabel("Resolved / received")  # Labels the proportional vertical axis.
plt.ylim(0, 1)  # Uses the complete logical range for a proportion.
plt.tight_layout()  # Keeps labels within the figure.
plt.show()  # Displays the final chart.


/var/folders/1j/v1lnwk6d5xv6pzqk_yc9xjcc0000gn/T/ipykernel_7494/3814596342.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Displays the final chart.


### 4. Produce a transparent process account


In [5]:
process_account = ["Loaded the synthetic course CSV without overwriting it.", "Removed only exact duplicate rows.", "Standardised documented city spelling variants.", "Converted numeric fields and exposed failures as missing.", "Excluded rows that failed the stated case-count rule from this descriptive summary.", "Calculated and plotted city-level resolution rates."]  # Records the steps needed to reproduce the result.
for step_number, process_step in enumerate(process_account, start=1):  # Numbers every process step in order.
    print(f"{step_number}. {process_step}")  # Displays the complete transparent process account.


1. Loaded the synthetic course CSV without overwriting it.
2. Removed only exact duplicate rows.
3. Standardised documented city spelling variants.
4. Converted numeric fields and exposed failures as missing.
5. Excluded rows that failed the stated case-count rule from this descriptive summary.
6. Calculated and plotted city-level resolution rates.


## Checks

Be ready to identify the unit of analysis, explain why invalid rows were retained for audit but excluded from this summary, state the denominator of the rate, and give at least two conclusions the chart does not justify.


## Next Steps

Use the separate mini-project template for your own question and dataset. Keep your raw data, code, result, limitations, and one meaningful debugging decision.
